# Week 01: Student Data Processing App

## Objective
Load `students.csv` from the **same folder as this notebook**, clean the data using Python functions, process it with NumPy, and save the cleaned data and statistical summary.

## Libraries
- **pandas** — CSV loading and tabular data cleaning
- **NumPy** — numerical calculations

> Keep `task.ipynb` and `students.csv` in the same folder when running this notebook.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

print("Libraries imported successfully.")
print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)


## 1. Load the CSV file

In [ ]:
# Locate students.csv in the same folder as this notebook.
# In Jupyter, Path.cwd() is the notebook's working directory.
DATA_FILE = Path.cwd() / "students.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE.name}. "
        "Place students.csv in the same folder as this notebook "
        "and run the notebook from that folder."
    )

data = pd.read_csv(DATA_FILE)

print("Loaded file:", DATA_FILE.resolve())
print("Original number of rows:", len(data))
display(data)


## 2. Inspect the dataset

In [ ]:
print("Dataset shape:", data.shape)

print("\nColumn data types:")
print(data.dtypes)

print("\nMissing values per column:")
print(data.isnull().sum())

print("\nDuplicate records:")
print(
    data.duplicated(
        subset=["name", "department", "score", "age"]
    ).sum()
)


## 3. Define data-cleaning functions

Each function performs one task. Keeping the tasks separate makes the code easier to test, reuse, and explain.


In [ ]:
def clean_text(data):
    """Remove extra spaces and standardize text capitalization."""
    data = data.copy()
    data["name"] = data["name"].str.strip().str.title()
    data["department"] = (
        data["department"].str.strip().str.title()
    )
    return data


def convert_types(data):
    """Convert numeric columns to numeric types."""
    data = data.copy()
    for column in ["student_id", "score", "age"]:
        data[column] = pd.to_numeric(
            data[column], errors="coerce"
        )
    return data


def remove_duplicates(data):
    """Remove duplicate student records."""
    return data.drop_duplicates(
        subset=["name", "department", "score", "age"],
        keep="first",
    ).copy()


def clean_missing_values(data):
    """Fill missing scores with the median and ages with the mean."""
    data = data.copy()

    valid_scores = data.loc[
        data["score"].between(0, 100), "score"
    ]
    median_score = valid_scores.median()

    if pd.isna(median_score):
        raise ValueError("No valid scores are available.")

    data["score"] = data["score"].fillna(median_score)

    mean_age = data["age"].mean()
    if pd.isna(mean_age):
        raise ValueError("No valid ages are available.")

    data["age"] = data["age"].fillna(round(mean_age))
    return data


def remove_invalid_scores(data):
    """Keep scores between 0 and 100, inclusive."""
    return data.loc[data["score"].between(0, 100)].copy()


print("Cleaning functions defined successfully.")


## 4. Clean the data

The steps are applied in order: standardize text, convert types, remove duplicate records, fill missing values, and remove records with scores outside the valid range of 0–100.


In [ ]:
cleaned = clean_text(data)
cleaned = convert_types(cleaned)

rows_before = len(cleaned)
cleaned = remove_duplicates(cleaned)
duplicates_removed = rows_before - len(cleaned)

cleaned = clean_missing_values(cleaned)

rows_before = len(cleaned)
cleaned = remove_invalid_scores(cleaned)
invalid_records_removed = rows_before - len(cleaned)

print("Duplicate records removed:", duplicates_removed)
print("Invalid-score records removed:", invalid_records_removed)
print("Remaining rows:", len(cleaned))

display(cleaned.reset_index(drop=True))


## 5. Calculate statistics with NumPy

NumPy calculates the mean, median, population standard deviation, 90th percentile, minimum, and maximum score.


In [ ]:
scores = cleaned["score"].to_numpy(dtype=float)

if scores.size == 0:
    raise ValueError("Cannot calculate statistics on empty data.")

statistics = {
    "Number of students": int(scores.size),
    "Mean score": float(np.mean(scores)),
    "Median score": float(np.median(scores)),
    "Standard deviation": float(np.std(scores)),
    "90th percentile": float(np.percentile(scores, 90)),
    "Minimum score": float(np.min(scores)),
    "Maximum score": float(np.max(scores)),
}

print("STUDENT SCORE STATISTICS")
print("-" * 30)

for name, value in statistics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.2f}")
    else:
        print(f"{name}: {value}")


## 6. Save the results

The cleaned CSV and summary report are saved in an `output` folder beside this notebook.


In [ ]:
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_file = OUTPUT_DIR / "cleaned_students.csv"
summary_file = OUTPUT_DIR / "summary.txt"

cleaned.to_csv(cleaned_file, index=False)

with summary_file.open("w", encoding="utf-8") as file:
    file.write("STUDENT DATA PROCESSING SUMMARY\n")
    file.write("=" * 35 + "\n\n")

    for name, value in statistics.items():
        if isinstance(value, float):
            file.write(f"{name}: {value:.2f}\n")
        else:
            file.write(f"{name}: {value}\n")

print("Files saved:")
print(cleaned_file.resolve())
print(summary_file.resolve())


## 7. Verify the saved results

In [ ]:
print("Cleaned CSV preview:")
display(pd.read_csv(OUTPUT_DIR / "cleaned_students.csv").head())

print("\nSummary report:")
print((OUTPUT_DIR / "summary.txt").read_text(encoding="utf-8"))


## Conclusion

The notebook loads the existing CSV file from its own folder, standardizes text, converts numeric columns, removes duplicate records, handles missing values, and excludes invalid scores.

NumPy calculates descriptive statistics. The cleaned dataset and summary report are saved in an `output` subfolder.

This project demonstrates Python functions, pandas, NumPy, CSV processing, and reproducible data analysis.
